# 01: Data Quality, Schema Validation & Temporal Splitting
This notebook demonstrates loading raw transactional interactions, performing comprehensive data validation, checking sparsity, and applying a strict temporal split to prevent data leakage.

In [1]:
import sys, os
from pathlib import Path
import pandas as pd
import numpy as np

# Robust root resolver whether run from notebooks/ or workspace root
_curr = Path('.').resolve()
PROJECT_ROOT = _curr if (_curr / 'src').exists() else (_curr.parent if (_curr.parent / 'src').exists() else (_curr.parents[1] if len(_curr.parents) > 1 and (_curr.parents[1] / 'src').exists() else _curr))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Switch working directory to project root so all relative paths ('data/...', 'artifacts/...') resolve seamlessly
os.chdir(PROJECT_ROOT)

from src.data.loader import DataLoader
from src.data.validator import DataValidator
from src.data.preprocess import DataPreprocessor


## 1. Load Raw Transactions and Product Catalog

In [2]:
loader = DataLoader('data/raw/retail_events.csv', 'data/raw/product_catalog.csv')
events_df, catalog_df = loader.load_and_merge()
print(f"Loaded {len(events_df):,} events across {events_df['customer_id'].nunique():,} customers and {events_df['product_id'].nunique():,} products.")
events_df.head()


2026-10-10 10:48:25 [INFO] [data_loader:45] Loading raw interactions from data\raw\retail_events.csv...
2026-10-10 10:48:25 [INFO] [data_loader:52] Loaded 48968 raw interaction records.
2026-10-10 10:48:25 [INFO] [data_loader:61] Loading product catalog from data\raw\product_catalog.csv...
2026-10-10 10:48:25 [INFO] [data_loader:68] Loaded catalog with 800 products.
Loaded 48,968 events across 3,000 customers and 800 products.


,customer_id,product_id,transaction_id,timestamp,quantity,unit_price,category,brand,description,discount,event_type
0,20816,1322,NaN,2026-01-01 00:04:30,1,274.48,Electronics/Smartphones,ApexLiving,Premium Smartphones item by ApexLiving with hi...,0.0,add_to_cart
1,21164,1255,NaN,2026-01-01 00:30:09,0,83.95,Home/Decor,PrimeStyle,Premium Decor item by PrimeStyle with high-per...,0.0,view
2,21601,1461,NaN,2026-01-01 00:35:02,0,125.99,Electronics/Audio,Zenith,Premium Audio item by Zenith with high-perform...,0.0,view
3,21217,1668,502961.0,2026-01-01 00:35:37,1,93.84,Home/Kitchen,LuxeLine,Premium Kitchen item by LuxeLine with high-per...,0.0,purchase
4,21044,1785,NaN,2026-01-01 00:52:10,1,55.81,Apparel/Footwear,QuantumLab,Premium Footwear item by QuantumLab with high-...,0.0,add_to_cart


## 2. Execute Data Validation & Quality Checks

In [3]:
validator = DataValidator(output_report_path='outputs/data_quality_report.json')
clean_df, dq_report = validator.validate_and_clean(events_df)
print("Data Quality Report Summary:")
print(f"- Validated Rows: {dq_report['row_count']:,}")
print(f"- Interaction Sparsity: {dq_report['sparsity']*100:.3f}%")
print(f"- Event Distribution: {dq_report['event_distribution']}")


2026-10-10 10:48:30 [INFO] [data_validator:53] Initiating Data Quality and Validation checks...
2026-10-10 10:48:30 [INFO] [data_validator:137] Data Quality report written to outputs\data_quality_report.json
2026-10-10 10:48:30 [INFO] [data_validator:138] Valid dataset: 48968 rows, 3000 customers, 800 products, Sparsity: 97.9597%
Data Quality Report Summary:
- Validated Rows: 48,968
- Interaction Sparsity: 97.960%
- Event Distribution: {'view': 29265, 'add_to_cart': 12382, 'purchase': 7321}


## 3. Strict Temporal Train / Validation / Test Split

In [4]:
preprocessor = DataPreprocessor(train_ratio=0.70, val_ratio=0.15, test_ratio=0.15)
train_df, val_df, test_df, cutoffs = preprocessor.temporal_split(clean_df)
print(f"Train events: {len(train_df):,} ({train_df['timestamp'].min()} to {cutoffs['train_val_cutoff']})")
print(f"Val events  : {len(val_df):,} ({cutoffs['train_val_cutoff']} to {cutoffs['val_test_cutoff']})")
print(f"Test events : {len(test_df):,} ({cutoffs['val_test_cutoff']} to {test_df['timestamp'].max()})")


2026-10-10 10:48:33 [INFO] [preprocess:38] Executing strict temporal train/val/test data split...
2026-10-10 10:48:33 [INFO] [preprocess:56] Temporal Split Complete:
  Train : 34,277 events (2026-01-01 00:04:30 to 2026-05-25 01:25:53)
  Val   : 7,345 events (2026-05-25 01:32:22 to 2026-06-13 05:45:58)
  Test  : 7,346 events (2026-06-13 05:46:27 to 2026-06-29 23:55:17)
Train events: 34,277 (2026-01-01 00:04:30 to 2026-05-25 01:25:53)
Val events  : 7,345 (2026-05-25 01:25:53 to 2026-06-13 05:45:58)
Test events : 7,346 (2026-06-13 05:45:58 to 2026-06-29 23:55:17)
